# RepoCoder Studio — Stage 3: Python → Java Translation

**Goal:** translate Python functions into Java methods/classes using a small code language model.

Final structure selected from the proposal and Stage 1/Stage 2 learnings:

- **Train:** XLCoST Python→Java aligned pairs
- **Validation:** XLCoST validation split
- **Test:** XLCoST test split + optional TransCoder-style evaluation subset
- **Fine-tune:** Qwen2.5-Coder with LoRA
- **Baselines:** Qwen2.5-Coder pretrained, optional DeepSeek-Coder pretrained
- **Metrics:** Compilation Success, Execution Correctness when tests are available, CodeBLEU/CodeBLEU-lite

This notebook intentionally avoids CodeNet training for the first working implementation because CodeNet is large and requires heavier pair alignment/cleaning.


In [1]:
# ============================================================
# CODE BLOCK 1: Stable environment setup
# Run this once, then restart runtime if Colab asks for it.
# ============================================================

!pip install -q --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  scipy==1.11.4 \
  scikit-learn==1.4.2

!pip install -q --no-cache-dir \
  torch \
  transformers==4.44.2 \
  datasets==2.21.0 \
  accelerate==0.34.2 \
  peft==0.12.0 \
  evaluate==0.4.2 \
  huggingface_hub==0.25.2 \
  tqdm \
  sentencepiece \
  protobuf \
  sacrebleu

# Optional. This may fail depending on package resolver; notebook has fallback metric.
!pip install -q --no-cache-dir codebleu || true


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 182.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 207.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 241.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 135.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inequality 1.1.2 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
pointpats 2.5.5 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires scipy>=1.13, but you have

In [1]:
# ============================================================
# CODE BLOCK 2: Imports and configuration
# ============================================================

# -----------------------------
# Standard library imports
# -----------------------------
import os          # File and directory operations
import re          # Regular expressions for text cleaning/pattern matching
import gc          # Garbage collection (manual memory cleanup)
import ast         # Abstract Syntax Trees (Python syntax validation)
import json        # JSON serialization/deserialization
import math        # Mathematical operations
import time        # Timing utilities (sleep, performance measurement)
import random      # Random sampling (with reproducibility)
import shutil      # File operations (copy, move, delete)
import tempfile    # Temporary file handling
import subprocess  # Running shell commands
import textwrap    # Formatting long strings neatly
from datetime import datetime  # Timestamps for logs/results
from pathlib import Path       # Cleaner path handling

# -----------------------------
# Third-party libraries
# -----------------------------
import numpy as np              # Numerical operations
import pandas as pd             # Tabular data handling
from tqdm.auto import tqdm      # Progress bars for loops

import torch                    # PyTorch deep learning framework
import transformers             # Hugging Face Transformers library
import datasets                 # Hugging Face Datasets library
import accelerate               # Hugging Face Accelerate (multi-GPU/TPU training)
import huggingface_hub          # Hugging Face Hub integration

# -----------------------------
# Hugging Face dataset/model utilities
# -----------------------------
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,                   # Tokenizer loader
    AutoModelForCausalLM,            # Model loader for causal language modeling
    TrainingArguments,               # Training configuration
    Trainer,                         # High-level training loop
    DataCollatorForLanguageModeling, # Handles batching and masking
    set_seed                         # Ensures reproducibility
)

# -----------------------------
# Parameter-efficient fine-tuning (PEFT)
# -----------------------------
from peft import (
    LoraConfig,      # LoRA configuration
    get_peft_model,  # Wraps base model with LoRA adapters
    TaskType         # Defines task type (e.g., causal LM)
)

# -----------------------------
# Environment checks
# -----------------------------
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# -----------------------------
# Reproducibility setup
# -----------------------------
SEED = 42
set_seed(SEED)          # Hugging Face seed
random.seed(SEED)       # Python random seed
np.random.seed(SEED)    # NumPy random seed

# ------------------------------------------------------------
# Experiment switches
# ------------------------------------------------------------
DEMO_MODE = True   # Demo mode: smaller dataset sizes, faster runs

RUN_QWEN_BASELINE = True         # Run Qwen pretrained baseline
RUN_DEEPSEEK_BASELINE = False    # Skip DeepSeek baseline (VRAM heavy)
RUN_FINE_TUNING = True           # Enable fine-tuning experiments

# ------------------------------------------------------------
# Model names
# ------------------------------------------------------------
QWEN_MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
DEEPSEEK_MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

# ------------------------------------------------------------
# Dataset names/configs
# ------------------------------------------------------------
XL_COST_DATASET_NAME = "codeparrot/xlcost-text-to-code"
XL_COST_PY_CONFIG = "Python-program-level"  # Python source config
XL_COST_JAVA_CONFIG = "Java-program-level"  # Java target config

# ------------------------------------------------------------
# Size settings
# ------------------------------------------------------------
if DEMO_MODE:
    TRAIN_EXAMPLES = 1000        # Limit training examples for speed
    VALIDATION_EXAMPLES = 100    # Limit validation examples
    TEST_EXAMPLES = 30           # Small test set for quick evaluation
    NUM_GENERATIONS = 1          # Single generation per prompt
else:
    TRAIN_EXAMPLES = None        # Use full dataset
    VALIDATION_EXAMPLES = None   # Use full dataset
    TEST_EXAMPLES = None         # Use full dataset
    NUM_GENERATIONS = 1

# ------------------------------------------------------------
# Length and quality constraints
# ------------------------------------------------------------
MAX_PROMPT_LENGTH = 768          # Max tokens for prompt
MAX_TOTAL_LENGTH = 1024          # Combined input+output length
MAX_NEW_TOKENS = 768             # Prevent runaway generations

MIN_PY_CHARS = 10                # Minimum Python code length
MIN_JAVA_CHARS = 10              # Minimum Java code length
MAX_PY_CHARS = 5000              # Maximum Python code length
MAX_JAVA_CHARS = 7000            # Maximum Java code length

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------
OUTPUT_DIR = "/content/RepoCoderStudio_Stage3_XLCoST_Only"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "evaluation_results")
MODEL_DIR = os.path.join(OUTPUT_DIR, "qwen_stage3_xlcost_lora")

# Ensure directories exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Device setup (GPU if available, else CPU)
# ------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Final configuration printout
# ------------------------------------------------------------
print("DEMO_MODE:", DEMO_MODE)
print("TRAIN_EXAMPLES:", TRAIN_EXAMPLES)
print("VALIDATION_EXAMPLES:", VALIDATION_EXAMPLES)
print("TEST_EXAMPLES:", TEST_EXAMPLES)
print("NUM_GENERATIONS:", NUM_GENERATIONS)
print("OUTPUT_DIR:", OUTPUT_DIR)


torch: 2.11.0+cu128
transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.34.2
huggingface_hub: 0.25.2
cuda: True
gpu: Tesla T4
DEMO_MODE: True
TRAIN_EXAMPLES: 1000
VALIDATION_EXAMPLES: 100
TEST_EXAMPLES: 30
NUM_GENERATIONS: 1
OUTPUT_DIR: /content/RepoCoderStudio_Stage3_XLCoST_Only


In [2]:
# ============================================================
# CODE BLOCK 3: Utility functions
# ============================================================

def load_dataset_safe(dataset_name, config_name=None, split=None):
    """
    Safely loads a Hugging Face dataset.
    - Accepts dataset_name, optional config_name, and optional split.
    - Tries different loading signatures depending on which arguments are provided.
    - If loading fails (e.g., dataset not found, network error), prints an error message
      and returns None instead of crashing the notebook.
    """
    try:
        if config_name is not None and split is not None:
            return load_dataset(dataset_name, config_name, split=split)
        elif config_name is not None:
            return load_dataset(dataset_name, config_name)
        elif split is not None:
            return load_dataset(dataset_name, split=split)
        else:
            return load_dataset(dataset_name)
    except Exception as e:
        print(f"Failed to load dataset={dataset_name}, config={config_name}")
        print("Error:", repr(e))
        return None


def limit_dataset(ds, limit, name):
    """
    Restricts dataset size for demo mode or controlled experiments.
    - If dataset is None, returns None immediately.
    - If limit is None, uses the full dataset and prints its size.
    - Otherwise, selects the first 'limit' examples (up to dataset length).
    - Prints a message showing how many examples are used vs total.
    """
    if ds is None:
        return None
    if limit is None:
        print(f"{name}: using full dataset ({len(ds)} examples)")
        return ds
    n = min(limit, len(ds))
    print(f"{name}: using {n}/{len(ds)} examples")
    return ds.select(range(n))


def get_split_or_first(dataset_dict, preferred_split):
    """
    Retrieves a dataset split safely.
    - If dataset_dict is None, returns None.
    - If dataset_dict is already a Dataset (not a DatasetDict), returns it directly.
    - If preferred_split exists in DatasetDict, returns that split.
    - Otherwise, falls back to the first available split and prints a message.
    """
    if dataset_dict is None:
        return None
    if isinstance(dataset_dict, Dataset):
        return dataset_dict
    if preferred_split in dataset_dict:
        return dataset_dict[preferred_split]
    first_split = list(dataset_dict.keys())[0]
    print(f"Using split '{first_split}' because '{preferred_split}' was not found.")
    return dataset_dict[first_split]


def cleanup_memory():
    """
    Frees up memory resources.
    - Calls Python garbage collector (gc.collect).
    - If CUDA is available, empties GPU cache to prevent out-of-memory errors.
    - Useful after unloading models or large datasets.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [3]:
# ============================================================
# CODE BLOCK 4: Load XLCoST Python and Java program-level configs
# ============================================================

# ------------------------------------------------------------
# Load XLCoST Python program-level dataset
# ------------------------------------------------------------
# This dataset contains Python source programs from XLCoST.
# The config "Python-program-level" specifies that we want full program-level examples.
xlcost_python_raw = load_dataset_safe(
    XL_COST_DATASET_NAME,
    config_name=XL_COST_PY_CONFIG
)

# ------------------------------------------------------------
# Load XLCoST Java program-level dataset
# ------------------------------------------------------------
# This dataset contains Java target programs from XLCoST.
# The config "Java-program-level" specifies that we want full program-level examples.
xlcost_java_raw = load_dataset_safe(
    XL_COST_DATASET_NAME,
    config_name=XL_COST_JAVA_CONFIG
)

# ------------------------------------------------------------
# Error handling: stop execution if either dataset failed to load
# ------------------------------------------------------------
if xlcost_python_raw is None:
    raise RuntimeError("Could not load XLCoST Python-program-level config.")

if xlcost_java_raw is None:
    raise RuntimeError("Could not load XLCoST Java-program-level config.")

# ------------------------------------------------------------
# Print dataset objects for confirmation
# ------------------------------------------------------------
print("XLCoST Python:", xlcost_python_raw)
print("XLCoST Java:", xlcost_java_raw)

# ------------------------------------------------------------
# Inspect available splits (e.g., train/validation/test)
# ------------------------------------------------------------
print("\nPython splits:", list(xlcost_python_raw.keys()))
print("Java splits:", list(xlcost_java_raw.keys()))

# ------------------------------------------------------------
# Inspect schema (column names) for the first split
# ------------------------------------------------------------
first_split = list(xlcost_python_raw.keys())[0]

print("\nPython columns:", xlcost_python_raw[first_split].column_names)
print("Java columns:", xlcost_java_raw[first_split].column_names)

# ------------------------------------------------------------
# Print sample rows for sanity check
# ------------------------------------------------------------
print("\nSample Python row:")
print(xlcost_python_raw[first_split][0])

print("\nSample Java row:")
print(xlcost_java_raw[first_split][0])


Generating train split:   0%|          | 0/9263 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/887 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/472 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/9623 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/911 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/494 [00:00<?, ? examples/s]

XLCoST Python: DatasetDict({
    train: Dataset({
        features: ['text', 'code'],
        num_rows: 9263
    })
    test: Dataset({
        features: ['text', 'code'],
        num_rows: 887
    })
    validation: Dataset({
        features: ['text', 'code'],
        num_rows: 472
    })
})
XLCoST Java: DatasetDict({
    train: Dataset({
        features: ['text', 'code'],
        num_rows: 9623
    })
    test: Dataset({
        features: ['text', 'code'],
        num_rows: 911
    })
    validation: Dataset({
        features: ['text', 'code'],
        num_rows: 494
    })
})

Python splits: ['train', 'test', 'validation']
Java splits: ['train', 'test', 'validation']

Python columns: ['text', 'code']
Java columns: ['text', 'code']

Sample Python row:
{'text': 'Maximum Prefix Sum possible by merging two given arrays | Python3 implementation of the above approach ; Stores the maximum prefix sum of the array A [ ] ; Traverse the array A [ ] ; Stores the maximum prefix sum of the arra

In [8]:
# ============================================================
# CODE BLOCK 5: XLCoST normalization and alignment helpers
# ============================================================

def clean_code_text(code):
    if code is None:
        return ""

    code = str(code)
    code = code.replace("```python", "")
    code = code.replace("```java", "")
    code = code.replace("```", "")
    code = code.replace("NEW_LINE", "\n")
    code = code.replace("INDENT", "    ")
    code = code.replace("DEDENT", "")
    code = code.replace("▁", " ")

    code = re.sub(r"\s+\n", "\n", code)
    code = re.sub(r"\n\s+\n", "\n\n", code)

    return textwrap.dedent(code).strip()


def clean_text(text):
    if text is None:
        return ""
    text = str(text).replace("▁", " ")
    text = " ".join(text.split())
    return text.strip()


def normalize_problem_title(text):
    """
    XLCoST descriptions often look like:
    'Rod Cutting | Python3 implementation of ...'
    'Rod Cutting | Java Program for ...'

    Matching the full sentence is too strict.
    We match mainly on the problem title before '|'.
    """
    text = clean_text(text).lower()

    if "|" in text:
        text = text.split("|")[0]

    remove_phrases = [
        "python3 implementation of",
        "python implementation of",
        "java program for",
        "java implementation of",
        "program for",
        "implementation of",
        "write a program to",
        "write a function to",
        "given a",
        "given an",
    ]

    for phrase in remove_phrases:
        text = text.replace(phrase, " ")

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = " ".join(text.split())
    return text.strip()


def pick_code_column(example):
    preferred = ["code", "program", "snippet", "target", "output", "answer"]

    for key in preferred:
        if key in example and example[key] is not None:
            value = clean_code_text(example[key])
            if value:
                return value

    string_values = []
    for key, value in example.items():
        if isinstance(value, str):
            cleaned = clean_code_text(value)
            if len(cleaned) > 20:
                string_values.append((len(cleaned), cleaned))

    if string_values:
        string_values.sort(reverse=True)
        return string_values[0][1]

    return ""


def pick_nl_column(example):
    preferred = ["text", "nl", "description", "docstring", "comment", "instruction"]

    for key in preferred:
        if key in example and example[key] is not None:
            value = clean_text(example[key])
            if value:
                return value

    candidates = []
    for key, value in example.items():
        if isinstance(value, str):
            cleaned = clean_text(value)
            if 10 <= len(cleaned) <= 500:
                if "NEW_LINE" not in cleaned and "def " not in cleaned and "class " not in cleaned:
                    candidates.append((len(cleaned), cleaned))

    if candidates:
        candidates.sort()
        return candidates[0][1]

    return ""


def looks_like_python(code):
    code = clean_code_text(code)
    return any(marker in code for marker in [
        "def ", "import ", "return ", "print(", "if ", "for ", "while ", ":"
    ])


def looks_like_java(code):
    code = clean_code_text(code)
    return any(marker in code for marker in [
        "class ", "public ", "static ", "void ", "int ", "String",
        "boolean", "double", "System.out", ";"
    ])


def dataset_to_title_buckets(ds, language_name):
    buckets = {}

    for idx, row in enumerate(ds):
        nl_text = pick_nl_column(row)
        code = pick_code_column(row)
        key = normalize_problem_title(nl_text)

        if not key or not code:
            continue

        buckets.setdefault(key, []).append({
            "idx": idx,
            "nl_text": nl_text,
            "code": code,
            "raw": row
        })

    print(f"{language_name}: {sum(len(v) for v in buckets.values())} usable rows from {len(ds)} raw rows")
    print(f"{language_name}: {len(buckets)} unique normalized problem-title keys")
    return buckets


def make_xlcost_pairs_by_title(py_ds, java_ds, split_name):
    """
    Align Python and Java using normalized problem title.
    This is less strict than full-description matching and usually recovers more pairs.
    """
    py_buckets = dataset_to_title_buckets(py_ds, f"Python-{split_name}")
    java_buckets = dataset_to_title_buckets(java_ds, f"Java-{split_name}")

    common_keys = sorted(set(py_buckets.keys()) & set(java_buckets.keys()))
    print(f"{split_name}: matched title keys = {len(common_keys)}")

    rows = []
    pair_id = 0

    for key in common_keys:
        py_items = py_buckets[key]
        java_items = java_buckets[key]

        n_pairs = min(len(py_items), len(java_items))

        for i in range(n_pairs):
            py_item = py_items[i]
            java_item = java_items[i]

            python_code = clean_code_text(py_item["code"])
            java_code = clean_code_text(java_item["code"])
            nl_text = clean_text(py_item["nl_text"])

            instruction = (
                "Translate the following Python code to equivalent Java code. "
                "Preserve the algorithm and input-output behavior."
            )

            if nl_text:
                instruction += f"\nProblem description: {nl_text}"

            rows.append({
                "source_dataset": "XLCoST",
                "split": split_name,
                "task_id": f"{split_name}_{pair_id}",
                "alignment_key": key,
                "python_idx": py_item["idx"],
                "java_idx": java_item["idx"],
                "instruction": instruction,
                "python_code": python_code,
                "java_code": java_code,
                "tests_json": json.dumps([])
            })

            pair_id += 1

    print(f"{split_name}: created Python→Java pairs = {len(rows)}")
    return Dataset.from_list(rows)

In [9]:
# ============================================================
# CODE BLOCK 6: Build aligned XLCoST Python→Java pairs
# ============================================================

common_splits = [
    split for split in xlcost_python_raw.keys()
    if split in xlcost_java_raw
]

if len(common_splits) == 0:
    raise RuntimeError("No common splits found between XLCoST Python and Java configs.")

xlcost = DatasetDict()

for split in common_splits:
    paired_split = make_xlcost_pairs_by_title(
        xlcost_python_raw[split],
        xlcost_java_raw[split],
        split
    )

    if len(paired_split) > 0:
        xlcost[split] = paired_split

if len(xlcost) == 0:
    raise RuntimeError("No aligned Python-Java pairs were created.")

print("Aligned XLCoST Python→Java dataset:")
print(xlcost)

for split in xlcost.keys():
    print("\nSplit:", split)
    print("Rows:", len(xlcost[split]))
    print("Columns:", xlcost[split].column_names)

    sample = xlcost[split][0]
    print("\nSample aligned pair:")
    print("Alignment key:", sample["alignment_key"])
    print("Python idx:", sample["python_idx"])
    print("Java idx:", sample["java_idx"])
    print("\nInstruction:", sample["instruction"][:500])
    print("\nPython code:", sample["python_code"][:500])
    print("\nJava code:", sample["java_code"][:500])
    break

Python-train: 9263 usable rows from 9263 raw rows
Python-train: 7846 unique normalized problem-title keys
Java-train: 9623 usable rows from 9623 raw rows
Java-train: 8050 unique normalized problem-title keys
train: matched title keys = 7619
train: created Python→Java pairs = 9001
Python-test: 887 usable rows from 887 raw rows
Python-test: 750 unique normalized problem-title keys
Java-test: 911 usable rows from 911 raw rows
Java-test: 765 unique normalized problem-title keys
test: matched title keys = 748
test: created Python→Java pairs = 883
Python-validation: 472 usable rows from 472 raw rows
Python-validation: 366 unique normalized problem-title keys
Java-validation: 494 usable rows from 494 raw rows
Java-validation: 377 unique normalized problem-title keys
validation: matched title keys = 366
validation: created Python→Java pairs = 471
Aligned XLCoST Python→Java dataset:
DatasetDict({
    train: Dataset({
        features: ['source_dataset', 'split', 'task_id', 'alignment_key', 'pyt

In [10]:
# ============================================================
# CODE BLOCK 7: Dataset cleaning and sanity checks
# ============================================================

def valid_translation_example(example):
    py = clean_code_text(example["python_code"])
    java = clean_code_text(example["java_code"])

    if len(py) < MIN_PY_CHARS:
        return False
    if len(java) < MIN_JAVA_CHARS:
        return False
    if len(py) > MAX_PY_CHARS:
        return False
    if len(java) > MAX_JAVA_CHARS:
        return False

    if not looks_like_python(py):
        return False
    if not looks_like_java(java):
        return False

    return True


def clean_translation_dataset(ds, name):
    before = len(ds)

    ds = ds.filter(valid_translation_example)

    seen = set()
    keep_indices = []

    for i, row in enumerate(ds):
        dedupe_key = (
            row["alignment_key"],
            clean_code_text(row["python_code"])[:300],
            clean_code_text(row["java_code"])[:300]
        )

        if dedupe_key not in seen:
            seen.add(dedupe_key)
            keep_indices.append(i)

    ds = ds.select(keep_indices)

    after = len(ds)
    print(f"{name}: {before} -> {after} examples kept")
    return ds


for split in list(xlcost.keys()):
    xlcost[split] = clean_translation_dataset(
        xlcost[split],
        f"XLCoST-{split}"
    )

print("\nPost-cleaning split sizes:")
for split in xlcost.keys():
    print(split, len(xlcost[split]))

print("\nManual sanity check:")
sample = xlcost[list(xlcost.keys())[0]][0]
print("Alignment key:", sample["alignment_key"])
print("Problem:", sample["instruction"][:300])
print("\nPython:", sample["python_code"][:500])
print("\nJava:", sample["java_code"][:500])

Filter:   0%|          | 0/9001 [00:00<?, ? examples/s]

XLCoST-train: 9001 -> 8945 examples kept


Filter:   0%|          | 0/883 [00:00<?, ? examples/s]

XLCoST-test: 883 -> 875 examples kept


Filter:   0%|          | 0/471 [00:00<?, ? examples/s]

XLCoST-validation: 471 -> 468 examples kept

Post-cleaning split sizes:
train 8945
test 875
validation 468

Manual sanity check:
Alignment key: 0
Problem: Translate the following Python code to equivalent Java code. Preserve the algorithm and input-output behavior.
Problem description: 0 | Returns the maximum value that can be put in a knapsack of capacity W ; Base Case ; If weight of the nth item is more than Knapsack of capacity W , then this item c

Python: def knapSack ( W , wt , val , n ) :
      if n == 0 or W == 0 :
      return 0
  if ( wt [ n - 1 ] > W ) :
      return knapSack ( W , wt , val , n - 1 )
  else :
      return max ( val [ n - 1 ] + knapSack ( W - wt [ n - 1 ] , wt , val , n - 1 ) , knapSack ( W , wt , val , n - 1 ) )
   val = [ 60 , 100 , 120 ]
 wt = [ 10 , 20 , 30 ]
 W = 50
 n = len ( val )
 print knapSack ( W , wt , val , n )

Java: class Knapsack { static int max ( int a , int b ) { return ( a > b ) ? a : b ; } static int knapSack ( int W , int wt [ ] , int va

In [13]:
# ============================================================
# CODE BLOCK 8: Split strategy + dataset sufficiency checks
# ============================================================

if all(k in xlcost for k in ["train", "validation", "test"]):
    train_data = xlcost["train"]
    validation_data = xlcost["validation"]
    xlcost_test_data = xlcost["test"]

elif all(k in xlcost for k in ["train", "test"]):
    temp = xlcost["train"].train_test_split(
        test_size=0.1,
        seed=SEED
    )

    train_data = temp["train"]
    validation_data = temp["test"]
    xlcost_test_data = xlcost["test"]

else:
    only_split = xlcost[list(xlcost.keys())[0]]

    temp = only_split.train_test_split(
        test_size=0.2,
        seed=SEED
    )

    train_data = temp["train"]

    temp2 = temp["test"].train_test_split(
        test_size=0.5,
        seed=SEED
    )

    validation_data = temp2["train"]
    xlcost_test_data = temp2["test"]


train_data = limit_dataset(
    train_data.shuffle(seed=SEED),
    TRAIN_EXAMPLES,
    "XLCoST train"
)

validation_data = limit_dataset(
    validation_data.shuffle(seed=SEED),
    VALIDATION_EXAMPLES,
    "XLCoST validation"
)

xlcost_test_data = limit_dataset(
    xlcost_test_data.shuffle(seed=SEED),
    TEST_EXAMPLES,
    "XLCoST test"
)

print("\nFinal dataset sizes:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("XLCoST Test:", len(xlcost_test_data))

print("\nTrain source distribution:")
print(pd.Series(train_data["source_dataset"]).value_counts())


MIN_TRAIN_FOR_FINE_TUNING = 500
MIN_VALIDATION_FOR_FINE_TUNING = 50
MIN_TEST_FOR_REPORTING = 30

dataset_sufficiency = {
    "train_examples": len(train_data),
    "validation_examples": len(validation_data),
    "test_examples": len(xlcost_test_data),
    "min_train_for_fine_tuning": MIN_TRAIN_FOR_FINE_TUNING,
    "min_validation_for_fine_tuning": MIN_VALIDATION_FOR_FINE_TUNING,
    "min_test_for_reporting": MIN_TEST_FOR_REPORTING,
    "enough_for_fine_tuning": (
        len(train_data) >= MIN_TRAIN_FOR_FINE_TUNING
        and len(validation_data) >= MIN_VALIDATION_FOR_FINE_TUNING
    ),
    "enough_for_strong_reporting": len(xlcost_test_data) >= MIN_TEST_FOR_REPORTING
}

print("\nDataset sufficiency check:")
print(json.dumps(dataset_sufficiency, indent=2))

if not dataset_sufficiency["enough_for_fine_tuning"]:
    print(
        "\nWARNING: Dataset is not large enough for reliable fine-tuning. "
        "Baseline evaluation can continue, but fine-tuning results should not be reported as strong evidence."
    )

    if RUN_FINE_TUNING:
        print("Setting RUN_FINE_TUNING = False to avoid training on too little data.")
        RUN_FINE_TUNING = False

if not dataset_sufficiency["enough_for_strong_reporting"]:
    print(
        "\nWARNING: Test set is small. Report baseline metrics as demo evidence only, "
        "not as final model performance."
    )

XLCoST train: using 1000/8945 examples
XLCoST validation: using 100/468 examples
XLCoST test: using 30/875 examples

Final dataset sizes:
Train: 1000
Validation: 100
XLCoST Test: 30

Train source distribution:
XLCoST    1000
Name: count, dtype: int64

Dataset sufficiency check:
{
  "train_examples": 1000,
  "validation_examples": 100,
  "test_examples": 30,
  "min_train_for_fine_tuning": 500,
  "min_validation_for_fine_tuning": 50,
  "min_test_for_reporting": 30,
  "enough_for_fine_tuning": true,
  "enough_for_strong_reporting": true
}


In [14]:
# ============================================================
# CODE BLOCK 8B: Role-based prompt construction helpers
# ============================================================

def build_prompt(example):
    """
    Builds a Qwen-compatible role-based prompt for code translation.
    - Cleans Python source code from the dataset example.
    - Wraps instruction in role tokens (<|system|>, <|user|>, <|assistant|>).
    - System role defines assistant behavior:
        * Expert code translation assistant.
        * Translate Python → Java.
        * Return only Java code (no explanations, markdown, or comments).
    - User role provides the translation task and the Python code.
    - Assistant role is left open for model to generate Java code.
    """
    py_code = clean_code_text(example["python_code"])

    return f"""<|system|>
You are RepoCoder Studio, an expert code translation assistant.
Translate Python code into correct, compilable, idiomatic Java.
Return only Java code. Do not include explanations or markdown.

<|user|>
Translate the following Python code to Java.
Preserve the algorithm and behavior.

Python code:
{py_code}

<|assistant|>
"""


def build_training_text(example):
    """
    Builds training text for supervised fine-tuning.
    - Combines role-based prompt with ground-truth Java code.
    - Ensures Java code is cleaned and stripped of extra whitespace.
    - Result is a full training example with system, user, and assistant roles.
    """
    return build_prompt(example) + clean_code_text(example["java_code"])


In [15]:
# ============================================================
# CODE BLOCK 9: Add prompt fields
# ============================================================

def add_prompt_fields(example):
    """
    Augments each dataset example with additional fields required for training:
    - 'prompt': the structured role-based input (system + user + assistant start).
    - 'training_text': the full supervised fine-tuning text (prompt + ground-truth output).
    This ensures consistency across training, validation, and evaluation datasets.
    """
    # Build the structured prompt for the example
    example["prompt"] = build_prompt(example)

    # Build the full training text (prompt + reference code/translation)
    example["training_text"] = build_training_text(example)

    return example


# ------------------------------------------------------------
# Apply prompt-building function to all dataset splits
# ------------------------------------------------------------

# Training split: used for fine-tuning the model
train_data = train_data.map(add_prompt_fields)

# Validation split: used for monitoring performance during training
validation_data = validation_data.map(add_prompt_fields)

# XLCoST test split: used for evaluation of translation quality
xlcost_test_data = xlcost_test_data.map(add_prompt_fields)

# ------------------------------------------------------------
# Sanity check: print sample prompt and training text
# ------------------------------------------------------------
# This helps confirm that the prompt and training_text fields
# were added correctly and formatted as expected.
print("Sample training prompt:")
print(train_data[0]["prompt"][:1500])  # Show first 1500 characters of prompt

print("\nSample training text:")
print(train_data[0]["training_text"][:1500])  # Show first 1500 characters of training text


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Sample training prompt:
<|system|>
You are RepoCoder Studio, an expert code translation assistant.
Translate Python code into correct, compilable, idiomatic Java.
Return only Java code. Do not include explanations or markdown.

<|user|>
Translate the following Python code to Java.
Preserve the algorithm and behavior.

Python code:
def minimumTimeToConvertString ( S , N ) :
      S = [ i for i in S ]
 upper = 0
 lower = 0
 for i in range ( N ) :
      c = S [ i ]
 if ( c . isupper ( ) ) :
      upper += 1
  else :
      lower += 1
   moves = 0
 if ( upper > N // 2 ) :
      i = 0
 while ( upper > N // 2 and i < N ) :
      if ( S [ i ] . isupper ( ) ) :
      S [ i ] += 32
 moves += 1
 upper -= 1
 lower += 1
  i += 1
   elif ( lower > N // 2 ) :
      i = 0
 while ( lower > N // 2 and i < N ) :
      if ( S [ i ] . islower ( ) ) :
      S [ i ] = chr ( ord ( S [ i ] ) - 32 )
 moves += 1
 upper += 1
 lower -= 1
  i += 1
   print ( moves )
 print ( " " . join ( S ) )
  if __name__ == ' _ 

In [16]:
# ============================================================
# CODE BLOCK 10: Token length statistics
# ============================================================

# ------------------------------------------------------------
# Load tokenizer for statistics
# ------------------------------------------------------------
# We use the same tokenizer as the training model (Qwen).
# trust_remote_code=True allows custom tokenizer implementations.
tokenizer_for_stats = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)

# Ensure pad_token is set (some models don't define it).
# If missing, reuse eos_token as the pad token.
if tokenizer_for_stats.pad_token is None:
    tokenizer_for_stats.pad_token = tokenizer_for_stats.eos_token


def token_len(text):
    """
    Utility function: compute token length of a given text.
    - Converts text to string.
    - Tokenizes without adding special tokens.
    - Returns number of tokens (length of input_ids).
    """
    return len(tokenizer_for_stats(str(text), add_special_tokens=False)["input_ids"])


# ------------------------------------------------------------
# Sample subset of training data for statistics
# ------------------------------------------------------------
# Limit to 500 examples or fewer if dataset is smaller.
sample_n = min(500, len(train_data))
sample = train_data.select(range(sample_n))

# ------------------------------------------------------------
# Compute token lengths for different fields
# ------------------------------------------------------------
# Prompt length: role-based structured input only.
prompt_lengths = [token_len(x["prompt"]) for x in sample]

# Full length: prompt + reference code (training_text).
full_lengths = [token_len(x["training_text"]) for x in sample]

# Java length: reference Java code only.
java_lengths = [token_len(x["java_code"]) for x in sample]

# ------------------------------------------------------------
# Aggregate statistics
# ------------------------------------------------------------
stats = {
    "sampled_examples": sample_n,  # number of examples used for stats
    "max_total_length": MAX_TOTAL_LENGTH,  # configured max length (prompt + output)
    "avg_prompt_tokens": float(np.mean(prompt_lengths)),  # average prompt length
    "avg_java_tokens": float(np.mean(java_lengths)),      # average Java code length
    "avg_full_tokens": float(np.mean(full_lengths)),      # average full training text length
    "max_full_tokens": int(np.max(full_lengths)),         # longest training text observed
    "truncation_rate_percent": float(
        np.mean([x > MAX_TOTAL_LENGTH for x in full_lengths]) * 100
    )  # percentage of examples exceeding max length
}

# ------------------------------------------------------------
# Print stats in JSON format for readability
# ------------------------------------------------------------
print(json.dumps(stats, indent=2))

# Save stats for later use in experiment card
token_length_stats = stats


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{
  "sampled_examples": 500,
  "max_total_length": 1024,
  "avg_prompt_tokens": 276.418,
  "avg_java_tokens": 254.026,
  "avg_full_tokens": 530.444,
  "max_full_tokens": 2134,
  "truncation_rate_percent": 4.8
}


In [17]:
# ============================================================
# CODE BLOCK 11: Tokenization for fine-tuning
# ============================================================

# ------------------------------------------------------------
# Load tokenizer for Qwen model
# ------------------------------------------------------------
# trust_remote_code=True allows custom tokenizer implementations.
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)

# Ensure pad_token is defined (some models don't set it).
# If missing, reuse eos_token as the pad token.
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token


def tokenize_for_training(example):
    """
    Tokenizes a dataset example for supervised fine-tuning.
    - Uses the 'training_text' field (prompt + ground-truth code).
    - Applies truncation to enforce MAX_TOTAL_LENGTH.
    - Does not pad (padding handled later by DataCollator).
    - Creates 'labels' identical to 'input_ids' for causal LM training.
      This ensures the model learns to predict the next token.
    """
    tokenized = qwen_tokenizer(
        example["training_text"],
        truncation=True,
        max_length=MAX_TOTAL_LENGTH,
        padding=False
    )
    # Labels are a copy of input_ids (standard for causal LM training)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


# ------------------------------------------------------------
# Apply tokenization to train and validation datasets
# ------------------------------------------------------------
# remove_columns ensures only tokenized fields remain (input_ids, attention_mask, labels).
train_tokenized = train_data.map(
    tokenize_for_training,
    remove_columns=train_data.column_names
)

validation_tokenized = validation_data.map(
    tokenize_for_training,
    remove_columns=validation_data.column_names
)

# ------------------------------------------------------------
# Collect tokenization statistics
# ------------------------------------------------------------
tokenization_stats = {
    "train_examples": len(train_tokenized),       # number of training examples
    "validation_examples": len(validation_tokenized),  # number of validation examples
    "max_total_length": MAX_TOTAL_LENGTH          # enforced max sequence length
}

# Print stats for quick verification
print(tokenization_stats)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'train_examples': 1000, 'validation_examples': 100, 'max_total_length': 1024}


In [18]:
# ============================================================
# CODE BLOCK 12: Java generation, extraction, compilation, metrics
# ============================================================

def strip_generation_artifacts(text):
    if text is None:
        return ""

    text = str(text)

    bad_markers = [
        "<|assistant|>", "<|user|>", "<|system|>",
        "<|im_start|>", "<|im_end|>",
        "<|endoftext|>",
        "```java", "```python", "```",
        "### Instruction:", "### Response:"
    ]

    for marker in bad_markers:
        text = text.replace(marker, "")

    return text.strip()


def extract_balanced_java_class(text):
    text = strip_generation_artifacts(text)

    class_match = re.search(r"(public\s+)?class\s+([A-Za-z_][A-Za-z0-9_]*)", text)

    if not class_match:
        return text.strip()

    start = class_match.start()
    brace_start = text.find("{", class_match.end())

    if brace_start == -1:
        return text[start:].strip()

    depth = 0
    end = None

    for i in range(brace_start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end is not None:
        return text[start:end].strip()

    return text[start:].strip()


def extract_java_code(generated_text):
    text = strip_generation_artifacts(generated_text)

    stop_markers = [
        "\nExplanation:",
        "\nThe Java code",
        "\nThis Java code",
        "\nIn this Java",
        "\nHere is",
        "\nNote:",
        "\nPython code:",
        "\nTranslate the following"
    ]

    for marker in stop_markers:
        pos = text.find(marker)
        if pos != -1:
            text = text[:pos]

    if "class " in text:
        return extract_balanced_java_class(text)

    return text.strip()


def detect_public_class_name(java_code):
    match = re.search(r"public\s+class\s+([A-Za-z_][A-Za-z0-9_]*)", java_code)

    if match:
        return match.group(1)

    match = re.search(r"class\s+([A-Za-z_][A-Za-z0-9_]*)", java_code)

    if match:
        return match.group(1)

    return "Main"


def ensure_java_compilation_unit(java_code):
    java_code = extract_java_code(java_code)

    if "class " in java_code:
        class_name = detect_public_class_name(java_code)
        return java_code, class_name

    wrapped = f"""
public class Main {{
{java_code}
}}
"""
    return wrapped.strip(), "Main"


def compile_java_code(java_code, timeout=10):
    java_code, class_name = ensure_java_compilation_unit(java_code)

    with tempfile.TemporaryDirectory() as tmpdir:
        java_path = os.path.join(tmpdir, f"{class_name}.java")

        with open(java_path, "w", encoding="utf-8") as f:
            f.write(java_code)

        try:
            result = subprocess.run(
                ["javac", java_path],
                capture_output=True,
                text=True,
                timeout=timeout
            )

            return {
                "compiles": result.returncode == 0,
                "stderr": result.stderr,
                "compiled_code": java_code,
                "class_name": class_name
            }

        except subprocess.TimeoutExpired:
            return {
                "compiles": False,
                "stderr": "Compilation timed out",
                "compiled_code": java_code,
                "class_name": class_name
            }


def simple_codebleu_lite(prediction, reference):
    pred_tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|[{}();=+\-*/<>]", str(prediction))
    ref_tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|[{}();=+\-*/<>]", str(reference))

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    pred_set = set(pred_tokens)
    ref_set = set(ref_tokens)

    overlap = len(pred_set & ref_set)
    precision = overlap / max(len(pred_set), 1)
    recall = overlap / max(len(ref_set), 1)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def generate_java(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            temperature=None,
            top_p=None,
            top_k=None
        )

    new_tokens = output_ids[0][input_len:]

    decoded = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return extract_java_code(decoded)


def evaluate_model_on_translation_dataset(
    model,
    tokenizer,
    dataset,
    dataset_name,
    model_label
):
    rows = []

    for example in tqdm(dataset, desc=f"{model_label} on {dataset_name}"):
        prediction_java = generate_java(
            model=model,
            tokenizer=tokenizer,
            prompt=example["prompt"]
        )

        reference_java = clean_code_text(example["java_code"])

        compile_result = compile_java_code(prediction_java)
        score = simple_codebleu_lite(prediction_java, reference_java)

        rows.append({
            "model": model_label,
            "dataset": dataset_name,
            "task_id": example["task_id"],
            "alignment_key": example.get("alignment_key", ""),
            "python_idx": example.get("python_idx", None),
            "java_idx": example.get("java_idx", None),
            "python_code": example["python_code"],
            "reference_java": reference_java,
            "prediction_java": prediction_java,
            "compiled_code": compile_result["compiled_code"],
            "compiles": compile_result["compiles"],
            "compilation_error": compile_result["stderr"],
            "codebleu_lite": score
        })

    summary = {
        "model": model_label,
        "dataset": dataset_name,
        "num_examples": len(rows),
        "compilation_success": float(np.mean([r["compiles"] for r in rows])) if rows else np.nan,
        "codebleu_lite": float(np.mean([r["codebleu_lite"] for r in rows])) if rows else np.nan
    }

    return rows, summary

In [19]:
# ============================================================
# CODE BLOCK 13: Sanity check for Stage 3 evaluation utilities
# ============================================================

# IMPORTANT:
# Java compilation, extraction, and metric helpers are already defined in Code Block 12.
# Do not redefine compile_java_code(), simple_codebleu_lite(), or evaluate_model_on_translation_dataset() here.
# This block only verifies that those functions exist and behave correctly.

# ------------------------------------------------------------
# List of required Stage 3 functions that must be present.
# These are the core utilities for Java code extraction,
# compilation, generation, and evaluation.
# ------------------------------------------------------------
_required_stage3_functions = [
    "extract_java_code",                  # Cleans and extracts valid Java code from generated text
    "compile_java_code",                  # Compiles Java code using javac and returns diagnostics
    "simple_codebleu_lite",               # Lightweight lexical similarity metric
    "generate_java",                      # Generates Java code from a model given a prompt
    "evaluate_model_on_translation_dataset" # Full evaluation harness for translation datasets
]

# ------------------------------------------------------------
# Check if any of the required functions are missing from globals().
# globals() is a dictionary of all currently defined names in the runtime.
# If any function is missing, raise a RuntimeError with a clear message.
# ------------------------------------------------------------
missing = [name for name in _required_stage3_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Missing required Stage 3 functions from Code Block 12: {missing}")

# ------------------------------------------------------------
# Run a test compilation to validate compile_java_code().
# We use a trivial Java program with a Main class and empty main method.
# ------------------------------------------------------------
_test_compile_result = compile_java_code("public class Main { public static void main(String[] args) { } }")

# ------------------------------------------------------------
# Ensure compile_java_code() returns a dictionary with diagnostic info.
# If it incorrectly returns a boolean (True/False), raise an error.
# This prevents silent failures if Code Block 12 was overwritten.
# ------------------------------------------------------------
if not isinstance(_test_compile_result, dict):
    raise RuntimeError(
        "compile_java_code() is returning a boolean instead of a dictionary. "
        "Re-run the corrected Code Block 12 and make sure no later block overwrites it."
    )

# ------------------------------------------------------------
# If all checks pass, print confirmation and show dictionary keys.
# This confirms the evaluation utilities are valid and ready for Stage 3.
# ------------------------------------------------------------
print("Stage 3 evaluation utilities are valid.")
print("compile_java_code() return keys:", list(_test_compile_result.keys()))


Stage 3 evaluation utilities are valid.
compile_java_code() return keys: ['compiles', 'stderr', 'compiled_code', 'class_name']


In [20]:
# ============================================================
# CODE BLOCK 14: Model loading utilities
# ============================================================

def load_model_and_tokenizer(model_name):
    """
    Loads a Hugging Face causal language model and tokenizer.
    - Ensures tokenizer has a pad token.
    - Loads model with appropriate dtype (fp16 if GPU available, else fp32).
    - Uses device_map="auto" for efficient placement.
    - Sets model to evaluation mode.

    Args:
        model_name (str): Hugging Face model identifier.

    Returns:
        (model, tokenizer): Loaded model and tokenizer objects.
    """
    print(f"Loading: {model_name}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # Ensure pad token exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model with dtype and device mapping
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )

    # Set to evaluation mode (disables dropout, etc.)
    model.eval()
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    model.generation_config.do_sample = False

    return model, tokenizer


def unload_model(model=None, tokenizer=None):
    """
    Unloads model and tokenizer to free memory.
    - Deletes objects.
    - Runs garbage collection.
    - Clears CUDA cache if GPU is available.

    Args:
        model: Model object to unload.
        tokenizer: Tokenizer object to unload.
    """
    try:
        del model
    except Exception:
        pass

    try:
        del tokenizer
    except Exception:
        pass

    # Force garbage collection
    gc.collect()

    # Clear GPU memory if available
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Model unloaded.")


In [21]:
# ============================================================
# CODE BLOCK 15: Qwen pretrained baseline
# ============================================================

# Reset these before every baseline run to avoid stale notebook-state results.
all_results = []
all_summaries = []

if RUN_QWEN_BASELINE:
    qwen_base_model, qwen_base_tokenizer = load_model_and_tokenizer(
        QWEN_MODEL_NAME
    )

    qwen_xlcost_results, qwen_xlcost_summary = evaluate_model_on_translation_dataset(
        qwen_base_model,
        qwen_base_tokenizer,
        xlcost_test_data,
        "XLCoST-Test",
        "Qwen-Pretrained"
    )

    all_results.extend(qwen_xlcost_results)
    all_summaries.append(qwen_xlcost_summary)

    print("Qwen pretrained summaries:")
    print(json.dumps(qwen_xlcost_summary, indent=2))

    unload_model(qwen_base_model, qwen_base_tokenizer)
else:
    print("RUN_QWEN_BASELINE is False. Skipping Qwen baseline.")



Loading: Qwen/Qwen2.5-Coder-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen-Pretrained on XLCoST-Test:   0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarni

Qwen pretrained summaries:
{
  "model": "Qwen-Pretrained",
  "dataset": "XLCoST-Test",
  "num_examples": 30,
  "compilation_success": 0.43333333333333335,
  "codebleu_lite": 0.8546948988088447
}
Model unloaded.


In [22]:
# ============================================================
# CODE BLOCK 15B: Analyze Qwen pretrained XLCoST outputs
# ============================================================

# ------------------------------------------------------------
# Filter results for Qwen pretrained on XLCoST test dataset
# ------------------------------------------------------------
qwen_stage3_df = pd.DataFrame([
    r for r in all_results
    if r.get("model") == "Qwen-Pretrained"
    and r.get("dataset") == "XLCoST-Test"
])

print("Rows:", len(qwen_stage3_df))


def analyze_java_prediction(row):
    """
    Analyzes a single Java prediction row to identify common failure issues.

    Args:
        row (dict): A result row containing prediction, compiled code, and errors.

    Returns:
        list[str]: Tags describing issues found in the generated code or compilation error.
    """
    generated = str(row.get("prediction_java", ""))   # Model output
    compiled_code = str(row.get("compiled_code", "")) # Code after wrapping
    error = str(row.get("compilation_error", ""))     # Compiler error message

    issues = []

    # Empty generation
    if generated.strip() == "":
        issues.append("empty_generation")

    # Missing class wrapper in raw prediction
    if "class " not in generated:
        issues.append("missing_class_wrapper_in_prediction")

    # Missing class wrapper even after wrapping
    if "class " not in compiled_code:
        issues.append("missing_class_wrapper_after_wrapping")

    # Missing static method or main entry point
    if "public static void main" not in generated and "static " not in generated:
        issues.append("missing_static_method_or_main")

    # Prompt artifacts or markdown leakage
    if "```" in generated or "<|im_" in generated:
        issues.append("prompt_or_markdown_leakage")

    # Compiler error messages
    if "error:" in error.lower():
        issues.append("java_compiler_error")

    # Specific compiler error: symbol not found
    if "cannot find symbol" in error.lower():
        issues.append("cannot_find_symbol")

    # Syntax errors: expected tokens (e.g., missing semicolon)
    if "';' expected" in error.lower() or "expected" in error.lower():
        issues.append("syntax_expected_token")

    return issues


# ------------------------------------------------------------
# Apply failure analysis and display results
# ------------------------------------------------------------
if len(qwen_stage3_df) > 0:
    # Apply analysis function to each row
    qwen_stage3_df["issue_tags"] = qwen_stage3_df.apply(
        analyze_java_prediction,
        axis=1
    )

    # Explode issue tags into separate rows for counting
    exploded = qwen_stage3_df.explode("issue_tags")

    # Show issue counts
    print("\nIssue counts:")
    display(
        exploded["issue_tags"]
        .value_counts()
        .reset_index()
        .rename(columns={"index": "issue", "issue_tags": "count"})
    )

    # Show compilation success/failure distribution
    print("\nCompilation success distribution:")
    display(
        qwen_stage3_df["compiles"]
        .value_counts(dropna=False)
        .reset_index()
    )

    # Show sample aligned references and predictions
    print("\nSample aligned references and predictions:")
    display(
        qwen_stage3_df[
            [
                "task_id",
                "python_code",
                "reference_java",
                "prediction_java",
                "compiles",
                "compilation_error",
                "codebleu_lite"
            ]
        ].head(5)
    )
else:
    print("No Qwen pretrained XLCoST results found.")


Rows: 30

Issue counts:


,count,count
0,java_compiler_error,17
1,cannot_find_symbol,6



Compilation success distribution:


,compiles,count
0,False,17
1,True,13



Sample aligned references and predictions:


,task_id,python_code,reference_java,prediction_java,compiles,compilation_error,codebleu_lite
0,test_330,"def findMaxAverage ( arr , n , k ) :\n if...",import java . io . * ; class GFG { static int ...,public class MaxAverageSubarray {\n public ...,True,,0.903846
1,test_238,def countTrailingZero ( x ) :\n count = 0...,import java . io . * ; class GFG { public stat...,public class CountTrailingZero {\n public s...,True,,0.888889
2,test_334,"M = 100\n def minAdjustmentCost ( A , n , targ...",import java . io . * ; import java . util . * ...,public class MinAdjustmentCost {\n public s...,False,/tmp/tmp8d7ryif0/MinAdjustmentCost.java:24: er...,0.608696
3,test_804,N = 4\n def func ( a ) :\n for i in range...,class GFG { static int N = 4 ; static void fun...,public class Main {\n public static void ma...,False,/tmp/tmpkds0tw6x/Main.java:29: error: not a st...,0.794521
4,test_253,def number_of_ways ( n ) :\n includes_3 =...,class GFG { static int number_of_ways ( int n ...,public class NumberWays {\n public static i...,True,,0.743590


In [23]:
# ============================================================
# CODE BLOCK 15C: Reference Java compilation quality check
# ============================================================

def check_reference_compilation_quality(dataset, sample_size=100, split_name="split"):
    n = min(sample_size, len(dataset))
    subset = dataset.shuffle(seed=SEED).select(range(n))

    rows = []

    for ex in tqdm(subset, desc=f"Checking reference Java compile quality: {split_name}"):
        result = compile_java_code(ex["java_code"])

        rows.append({
            "task_id": ex["task_id"],
            "alignment_key": ex["alignment_key"],
            "reference_compiles": result["compiles"],
            "reference_error": result["stderr"][:500],
            "java_code": ex["java_code"][:500]
        })

    df = pd.DataFrame(rows)

    print(f"\nReference Java compilation quality for {split_name}:")
    print("Checked:", len(df))
    print("Reference compile rate:", df["reference_compiles"].mean())

    display(
        df["reference_compiles"]
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={"index": "reference_compiles", "reference_compiles": "count"})
    )

    display(df[df["reference_compiles"] == False].head(5))

    return df


reference_train_quality_df = check_reference_compilation_quality(
    train_data if "train_data" in globals() else xlcost["train"],
    sample_size=100,
    split_name="train"
)

reference_test_quality_df = check_reference_compilation_quality(
    xlcost["test"],
    sample_size=100,
    split_name="test"
)

Checking reference Java compile quality: train:   0%|          | 0/100 [00:00<?, ?it/s]


Reference Java compilation quality for train:
Checked: 100
Reference compile rate: 0.62


,count,count
0,True,62
1,False,38


,task_id,alignment_key,reference_compiles,reference_error,java_code
0,train_8267,sorting all array elements except one,False,/tmp/tmp0ayqgo4n/GFG.java:1: error: cannot fin...,import java . util . Arrays ; class GFG { stat...
1,train_8572,sum of frequencies of characters of a string p...,False,/tmp/tmpx06dhw9r/GFG.java:1: error: cannot fin...,import java . util . HashSet ; class GFG { sta...
2,train_7851,reorder the position of the words in alphabeti...,False,/tmp/tmpzmyxiv4_/GFG.java:1: error: cannot fin...,import java . util . * ; class GFG { static vo...
3,train_3269,find the coordinates of the fourth vertex of a...,False,/tmp/tmpduwhiapk/GfG.java:1: error: unclosed c...,import java . util . HashMap ; import java . u...
5,train_6882,periodic binary string with minimum period and...,False,/tmp/tmpd8_phugi/GFG.java:1: error: ')' expect...,class GFG { static void findPeriodicString ( S...


Checking reference Java compile quality: test:   0%|          | 0/100 [00:00<?, ?it/s]


Reference Java compilation quality for test:
Checked: 100
Reference compile rate: 0.68


,count,count
0,True,68
1,False,32


,task_id,alignment_key,reference_compiles,reference_error,java_code
3,test_804,sort matrix in alternating ascending and desce...,False,/tmp/tmpk7elm27f/GFG.java:1: error: unclosed s...,class GFG { static int N = 4 ; static void fun...
6,test_179,count of all pairs in an array with minimum ab...,False,/tmp/tmpyhsq09ek/GFG.java:1: error: cannot fin...,import java . util . Arrays ; class GFG { stat...
15,test_151,cost to balance the parentheses,False,/tmp/tmpn693di01/GFG.java:1: error: unclosed c...,import java . io . * ; class GFG { static int ...
16,test_765,remove odd frequency characters from the string,False,/tmp/tmp7epiht32/GFG.java:1: error: cannot fin...,import java . util . * ; class GFG { static St...
17,test_226,count subarrays having sum modulo k same as th...,False,/tmp/tmpojdn95id/GFG.java:1: error: cannot fin...,import java . util . * ; import java . lang . ...


In [24]:
# ============================================================
# CODE BLOCK 16: Optional DeepSeek pretrained baseline
# ============================================================

# Run the DeepSeek baseline evaluation only if the flag is enabled
if RUN_DEEPSEEK_BASELINE:
    # Load the DeepSeek pretrained model and its tokenizer
    deepseek_model, deepseek_tokenizer = load_model_and_tokenizer(
        DEEPSEEK_MODEL_NAME
    )

    # Evaluate DeepSeek on the XLCoST test dataset (Python→Java translation tasks)
    deepseek_xlcost_results, deepseek_xlcost_summary = evaluate_model_on_translation_dataset(
        deepseek_model,
        deepseek_tokenizer,
        xlcost_test_data,        # Dataset containing Python prompts and Java references
        "XLCoST-Test",           # Dataset name for logging
        "DeepSeek-Pretrained"    # Model label for identification
    )

    # Extend global results and summaries with DeepSeek outputs
    all_results.extend(deepseek_xlcost_results)    # Add detailed per-example rows
    all_summaries.append(deepseek_xlcost_summary)  # Add dataset-level summary metrics

    # Print summaries specifically for DeepSeek pretrained baseline
    print("DeepSeek pretrained summaries:")
    for s in all_summaries:
        if s["model"] == "DeepSeek-Pretrained":    # Filter summaries by model label
            print(json.dumps(s, indent=2))         # Pretty-print JSON summary

    # Unload the model and tokenizer to free GPU/CPU memory resources
    unload_model(deepseek_model, deepseek_tokenizer)


In [25]:
# ============================================================
# CODE BLOCK 17: Fine-tune Qwen with LoRA
# ============================================================

if RUN_FINE_TUNING:
    # Clean up GPU/CPU memory before starting fine-tuning
    cleanup_memory()

    # Load the pretrained Qwen model with appropriate precision and device mapping
    qwen_ft_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,  # Use FP16 if GPU available
        device_map="auto" if torch.cuda.is_available() else None,                   # Auto device placement on GPU
        trust_remote_code=True                                                     # Allow custom model code
    )

    # Disable caching to avoid issues during training
    qwen_ft_model.config.use_cache = False

    # Configure LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,  # Task type: causal language modeling
        r=8,                           # Rank of LoRA matrices
        lora_alpha=16,                 # Scaling factor
        lora_dropout=0.05,             # Dropout for LoRA layers
        target_modules=[               # Target modules in transformer layers to apply LoRA
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]
    )

    # Wrap the base model with LoRA adapters
    qwen_ft_model = get_peft_model(qwen_ft_model, lora_config)
    # Print trainable parameters for verification
    qwen_ft_model.print_trainable_parameters()

    # Data collator: prepares batches for training (no masked language modeling here)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=qwen_tokenizer,
        mlm=False
    )

    # Training configuration
    training_args = TrainingArguments(
        output_dir=MODEL_DIR,                  # Directory to save checkpoints
        num_train_epochs=2 if DEMO_MODE else 3, # Fewer epochs in demo mode
        per_device_train_batch_size=1,         # Small batch size (VRAM-friendly)
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,         # Accumulate gradients for effective batch size
        learning_rate=2e-4,                    # Learning rate for fine-tuning
        weight_decay=0.01,                     # Regularization
        logging_steps=20,                      # Log training progress every 20 steps
        evaluation_strategy="steps",           # Evaluate periodically
        eval_steps=100,                        # Evaluate every 100 steps
        save_steps=100,                        # Save checkpoint every 100 steps
        save_total_limit=2,                    # Keep only 2 latest checkpoints
        fp16=torch.cuda.is_available(),        # Use FP16 if GPU available
        report_to="none",                      # Disable external logging (e.g., WandB)
        remove_unused_columns=False            # Keep all dataset columns
    )

    # Hugging Face Trainer setup
    trainer = Trainer(
        model=qwen_ft_model,
        args=training_args,
        train_dataset=train_tokenized,         # Tokenized training dataset
        eval_dataset=validation_tokenized,     # Tokenized validation dataset
        data_collator=data_collator,
        tokenizer=qwen_tokenizer
    )

    # Start fine-tuning
    trainer.train()

    # Save the fine-tuned model and tokenizer
    trainer.save_model(MODEL_DIR)
    qwen_tokenizer.save_pretrained(MODEL_DIR)

    print("Saved fine-tuned LoRA model to:", MODEL_DIR)


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss,Validation Loss
100,0.302200,0.289317
200,0.275300,0.276131


Saved fine-tuned LoRA model to: /content/RepoCoderStudio_Stage3_XLCoST_Only/qwen_stage3_xlcost_lora


In [27]:
# ============================================================
# CODE BLOCK 18: Inspect Stage 3 Python→Java generations
# ============================================================

def inspect_generation_results(results, n=3):
    if results is None or len(results) == 0:
        print("No results to inspect.")
        return

    sample_rows = results[:min(n, len(results))]

    for i, row in enumerate(sample_rows):
        print("=" * 100)
        print(f"EXAMPLE {i + 1}")
        print("Task ID:", row.get("task_id", "N/A"))
        print("Alignment key:", row.get("alignment_key", "N/A"))
        print("Compiles:", row.get("compiles", "N/A"))
        print("CodeBLEU-lite:", row.get("codebleu_lite", "N/A"))

        print("\nPYTHON SOURCE:")
        print(str(row.get("python_code", ""))[:1200])

        print("\nREFERENCE JAVA:")
        print(str(row.get("reference_java", ""))[:1200])

        print("\nPREDICTED JAVA:")
        print(str(row.get("prediction_java", ""))[:1200])

        print("\nCOMPILATION ERROR:")
        err = str(row.get("compilation_error", ""))
        print(err[:1200] if err.strip() else "None")

        print("=" * 100)


print("Inspecting Qwen Stage 3 baseline generations:")
inspect_generation_results(
    qwen_xlcost_results,
    n=3
)

Inspecting Qwen Stage 3 baseline generations:
EXAMPLE 1
Task ID: test_330
Alignment key: find maximum average subarray of k length
Compiles: True
CodeBLEU-lite: 0.9038461538461539

PYTHON SOURCE:
def findMaxAverage ( arr , n , k ) :
      if ( k > n ) :
      return - 1
  sum = arr [ 0 ]
 for i in range ( 1 , k ) :
      sum += arr [ i ]
  max_sum = sum
 max_end = k - 1
 for i in range ( k , n ) :
      sum = sum + arr [ i ] - arr [ i - k ]
 if ( sum > max_sum ) :
      max_sum = sum
 max_end = i
   return max_end - k + 1
  arr = [ 1 , 12 , - 5 , - 6 , 50 , 3 ]
 k = 4
 n = len ( arr )
 print ( " The   maximum   average   subarray   of   length " , k , " begins   at   index " , findMaxAverage ( arr , n , k ) )

REFERENCE JAVA:
import java . io . * ; class GFG { static int findMaxAverage ( int arr [ ] , int n , int k ) { if ( k > n ) return - 1 ; int sum = arr [ 0 ] ; for ( int i = 1 ; i < k ; i ++ ) sum += arr [ i ] ; int max_sum = sum , max_end = k - 1 ; for ( int i = k ; i < n ; i ++ 

In [28]:
# ============================================================
# CODE BLOCK 19: Save comparison artifacts
# ============================================================

# Convert all summaries (dataset-level metrics) into a DataFrame
summary_df = pd.DataFrame(all_summaries)

# Path to save the summary CSV file
summary_csv = os.path.join(
    RESULTS_DIR,
    "stage3_model_comparison_summary.csv"
)

# Save summary metrics to CSV (one row per model/dataset combination)
summary_df.to_csv(summary_csv, index=False)

# Path to save detailed predictions CSV file
detailed_csv = os.path.join(
    RESULTS_DIR,
    "stage3_detailed_predictions.csv"
)

# Save all per-example results (predictions, references, compilation results, scores)
pd.DataFrame(all_results).to_csv(detailed_csv, index=False)

# Build experiment card (JSON metadata for reproducibility and documentation)
experiment_card = {
    "stage": "Stage 3",  # Current stage of the experiment pipeline
    "task": "Python to Java Code Translation",
    "training_datasets": [
        "XLCoST Python-program-level + Java-program-level aligned by split/index"
    ],
    "validation_datasets": [
        "XLCoST validation split"
    ],
    "evaluation_datasets": [
        "XLCoST test split"
    ],
    "dataset_decision": (
        "The final Stage 3 implementation uses only XLCoST. Instead of loading a nonexistent "
        "single Python-Java translation mirror, the notebook loads codeparrot/xlcost-text-to-code "
        "with explicit Python-program-level and Java-program-level configs, then aligns rows by "
        "split and index to create Python-to-Java translation pairs. Other datasets were removed "
        "to keep the implementation reliable and reproducible on Colab."
    ),
    "models_compared": [
        "Qwen pretrained",
        "DeepSeek pretrained optional",
        "Qwen LoRA fine-tuned"
    ],
    "base_model_for_finetuning": QWEN_MODEL_NAME,
    "comparison_baseline": DEEPSEEK_MODEL_NAME,
    "metrics": [
        "Compilation Success",
        "CodeBLEU-lite"
    ],
    "demo_mode": DEMO_MODE,                  # Whether demo mode was enabled
    "train_examples": len(train_data),       # Number of training examples
    "validation_examples": len(validation_data), # Number of validation examples
    "test_examples": len(xlcost_test_data),  # Number of test examples
    "token_length_stats": token_length_stats, # Token length statistics
    "tokenization_stats": tokenization_stats, # Tokenization statistics
    "summaries": all_summaries,              # All dataset-level summaries
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S") # Save timestamp
}

# Path to save experiment card JSON
experiment_card_path = os.path.join(
    RESULTS_DIR,
    "stage3_experiment_card.json"
)

# Write experiment card to JSON file
with open(experiment_card_path, "w") as f:
    json.dump(experiment_card, f, indent=2)

# Print confirmation of saved artifacts
print("Saved:")
print(summary_csv)
print(detailed_csv)
print(experiment_card_path)

# Display summary DataFrame for quick inspection in notebook
display(summary_df)


Saved:
/content/RepoCoderStudio_Stage3_XLCoST_Only/evaluation_results/stage3_model_comparison_summary.csv
/content/RepoCoderStudio_Stage3_XLCoST_Only/evaluation_results/stage3_detailed_predictions.csv
/content/RepoCoderStudio_Stage3_XLCoST_Only/evaluation_results/stage3_experiment_card.json


,model,dataset,num_examples,compilation_success,codebleu_lite
0,Qwen-Pretrained,XLCoST-Test,30,0.433333,0.854695
1,Qwen-FineTuned-XLCoST,XLCoST-Test,30,0.500000,0.895245


In [32]:
# ============================================================
# CODE BLOCK 20: Final Python→Java demo function
# ============================================================

def build_translation_prompt_for_demo(python_code):
    return f"""You are RepoCoder Studio, an expert Python to Java code translation assistant.

Translate the following Python code into correct, compilable Java.
Preserve the algorithm and behavior.
Return only Java code.

Python code:
{python_code}

Java code:
"""


def translate_python_to_java_demo(
    python_code,
    model=None,
    tokenizer=None
):
    should_unload = False

    if model is None or tokenizer is None:
        model_name_or_path = (
            FT_MODEL_OUTPUT_DIR
            if RUN_FINE_TUNING and os.path.exists(FT_MODEL_OUTPUT_DIR)
            else QWEN_MODEL_NAME
        )

        model, tokenizer = load_model_and_tokenizer(model_name_or_path)
        should_unload = True

    prompt = build_translation_prompt_for_demo(python_code)

    result = generate_java(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS
    )

    if should_unload:
        unload_model(model, tokenizer)

    return result


sample_python = """
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(factorial(5))
"""


print("Python input:")
print(sample_python)

print("\nGenerated Java:")
if RUN_FINE_TUNING and "qwen_ft_model" in globals() and "qwen_tokenizer" in globals():
    print(
        translate_python_to_java_demo(
            sample_python,
            qwen_ft_model,
            qwen_tokenizer
        )
    )
else:
    print(translate_python_to_java_demo(sample_python))

Python input:

def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(factorial(5))


Generated Java:


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


class GFG { static int factorial ( int n ) { if ( n <= 1 ) return 1 ; return n * factorial ( n - 1 ) ; } public static void main ( String [ ] args ) { System . out . println ( factorial ( 5 ) ) ; } }
